In [1]:
import os
from PIL import Image
import numpy as np

In [2]:
# Parramètres :

folder_normals = "./images-blender/baked"
out_folder = "./images-blender"

In [3]:
def mix_normals(normal_map1, normal_map2):

    # Pixels nuls
    mask1_zero = np.all(normal_map1 == 0, axis=-1, keepdims=True)
    mask2_zero = np.all(normal_map2 == 0, axis=-1, keepdims=True)

    # Cas 1 : normal_map1 est nulle → prendre normal_map2
    # Cas 2 : normal_map2 est nulle → prendre normal_map1
    # Cas 3 : aucune n'est nulle → moyenne
    mixed = np.where(
        mask1_zero, # Cas 1 :
        normal_map2,
        np.where(
            mask2_zero, # Cas 2 :
            normal_map1,
            # Cas 3 :
            0.5 * (normal_map1 + normal_map2)
        )
    )

    return mixed


def merge_normals(folder_normals):

    im_names = sorted([
        f for f in os.listdir(folder_normals)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

    if not im_names:
        raise ValueError("Aucune image trouvée dans le dossier")
    print(f"Images trouvées : {im_names}")

    # Charger la première carte
    normal_map = np.array(
        Image.open(os.path.join(folder_normals, im_names[0])).convert("RGB"),
        dtype=np.float32
    ) / 255.0

    # Fusion avec les suivantes
    for name in im_names[1:]:
        next_normal_map = np.array(
            Image.open(os.path.join(folder_normals, name)).convert("RGB"),
            dtype=np.float32
        ) / 255.0
        normal_map = mix_normals(normal_map, next_normal_map)

    return normal_map

In [4]:
merged_normals = merge_normals(folder_normals)

output_image = Image.fromarray((merged_normals * 255).astype(np.uint8))
output_image.show()

output_image.save(os.path.join(out_folder, "normal_merged.png"))

Images trouvées : ['normal_0_bakedd.png', 'normal_1_bakedd.png', 'normal_2_bakedd.png', 'normal_3_bakedd.png', 'normal_4_bakedd.png', 'normal_5_bakedd.png', 'normal_6_bakedd.png', 'normal_7_bakedd.png', 'normal_8_bakedd.png', 'normal_9_bakedd.png']
